# ARIMA/SARIMA - Identificação, ajuste e previsão

Processa as três séries com SARIMA, identifica ordens via ACF/PACF e grid search (AIC),
ajusta apenas no treino e prevê os 28 dias da validação.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.statespace.sarimax import SARIMAX
from itertools import product
import itertools

## Carregamento de dados

In [ ]:
df_train = pd.read_csv("../../dados/treino.csv")
df_val = pd.read_csv("../../dados/validacao.csv")

df_train["date"] = pd.to_datetime(df_train["date"])
df_val["date"] = pd.to_datetime(df_val["date"])

series_names = ["store_total", "FOODS", "HOBBIES"]

print(f"Treino: {df_train['date'].min()} a {df_train['date'].max()} ({len(df_train)} dias)")
print(f"Validação: {df_val['date'].min()} a {df_val['date'].max()} ({len(df_val)} dias)")

## Grid search para SARIMA

Busca as melhores ordens (p, d, q, P, D, Q) usando AIC no treino.

In [ ]:
def find_best_sarima_order(serie, serie_name, m=7, max_p=2, max_d=1, max_q=2, max_P=1, max_D=1, max_Q=1):
    """
    Grid search para encontrar a melhor ordem SARIMA usando AIC.
    Limita a busca para ordens pequenas (computação rápida).
    """
    best_aic = np.inf
    best_order = None
    best_seasonal_order = None
    results_log = []
    
    p_range = range(0, max_p + 1)
    d_range = range(0, max_d + 1)
    q_range = range(0, max_q + 1)
    P_range = range(0, max_P + 1)
    D_range = range(0, max_D + 1)
    Q_range = range(0, max_Q + 1)
    
    total_combinations = len(list(p_range)) * len(list(d_range)) * len(list(q_range)) * \
                        len(list(P_range)) * len(list(D_range)) * len(list(Q_range))
    print(f"\n{serie_name}: testando {total_combinations} combinações...")
    
    count = 0
    for p, d, q, P, D, Q in itertools.product(p_range, d_range, q_range, P_range, D_range, Q_range):
        count += 1
        if count % 50 == 0:
            print(f"  Progresso: {count}/{total_combinations}")
        
        try:
            model = SARIMAX(
                serie,
                order=(p, d, q),
                seasonal_order=(P, D, Q, m),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            results = model.fit(disp=False, maxiter=200)
            aic = results.aic
            
            results_log.append({
                'order': (p, d, q),
                'seasonal_order': (P, D, Q, m),
                'aic': aic
            })
            
            if aic < best_aic:
                best_aic = aic
                best_order = (p, d, q)
                best_seasonal_order = (P, D, Q, m)
        except:
            continue
    
    # Exibe top 5 melhores ordens
    results_log_sorted = sorted(results_log, key=lambda x: x['aic'])
    print(f"\n  Top 5 melhores ordens:")
    for i, res in enumerate(results_log_sorted[:5], 1):
        print(f"    {i}. SARIMA{res['order']}{res['seasonal_order']}: AIC={res['aic']:.2f}")
    
    print(f"\n  ✓ Melhor ordem: SARIMA{best_order}{best_seasonal_order} com AIC={best_aic:.2f}")
    
    return best_order, best_seasonal_order

## Identificação de ordens

In [ ]:
# Grid search para cada série
# Limita a p, q em {0, 1, 2} e P, Q em {0, 1} para rapidez
sarima_orders = {}

for serie_name in series_names:
    order, seasonal_order = find_best_sarima_order(
        df_train[serie_name].values,
        serie_name,
        m=7,
        max_p=2, max_d=1, max_q=2,
        max_P=1, max_D=1, max_Q=1
    )
    sarima_orders[serie_name] = (order, seasonal_order)

## Ajuste dos modelos SARIMA no treino

In [ ]:
# Ajusta o modelo final em cada série usando toda a série de treino
fitted_models = {}

for serie_name in series_names:
    order, seasonal_order = sarima_orders[serie_name]
    print(f"\nAjustando {serie_name}: SARIMA{order}{seasonal_order}...")
    
    model = SARIMAX(
        df_train[serie_name].values,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    results = model.fit(disp=False, maxiter=200)
    fitted_models[serie_name] = results
    
    print(f"  AIC: {results.aic:.2f}")
    print(f"  BIC: {results.bic:.2f}")

## Previsão para o horizonte de validação (28 dias)

In [ ]:
# Faz previsão para os 28 dias da validação
h = len(df_val)  # 28 dias
previsoes_sarima = {}

for serie_name in series_names:
    results = fitted_models[serie_name]
    # forecast: previsão puntual dos próximos h períodos
    forecast = results.get_forecast(steps=h).predicted_mean.values
    previsoes_sarima[serie_name] = forecast
    
    print(f"{serie_name}: min={forecast.min():.0f}, max={forecast.max():.0f}, mean={forecast.mean():.0f}")

## Exportação das previsões no formato requerido

In [ ]:
# Cria DataFrame no formato: date, series, modelo, yhat
predictions_list = []

for serie_name in series_names:
    for i, (date, yhat) in enumerate(zip(df_val["date"], previsoes_sarima[serie_name])):
        predictions_list.append({
            "date": date,
            "series": serie_name,
            "modelo": "sarima",
            "yhat": yhat
        })

df_previsoes = pd.DataFrame(predictions_list)
print(f"\nPrevisões SARIMA geradas: {len(df_previsoes)} linhas")
print(df_previsoes.head(10))

## Visualização das previsões vs valores reais

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

for idx, serie_name in enumerate(series_names):
    ax = axes[idx]
    
    # Plot dos últimos 100 dias do treino
    tail_train = min(100, len(df_train))
    ax.plot(df_train["date"].iloc[-tail_train:], 
            df_train[serie_name].iloc[-tail_train:], 
            label="Treino (últimos 100 dias)", marker="o", markersize=3)
    
    # Plot da validação
    ax.plot(df_val["date"], df_val[serie_name], 
            label="Validação (real)", marker="s", markersize=5, color="orange")
    
    # Plot das previsões SARIMA
    ax.plot(df_val["date"], previsoes_sarima[serie_name], 
            label="SARIMA (predito)", marker="^", markersize=5, color="red", linestyle="--")
    
    ax.set_title(f"{serie_name} - SARIMA", fontsize=12, fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Vendas")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../../docs/previsoes_sarima.png", dpi=150, bbox_inches="tight")
print("\n✓ Gráfico de previsões salvo em docs/previsoes_sarima.png")
plt.show()

## Resumo das ordens identificadas

In [ ]:
print("\n" + "="*70)
print("ORDENS SARIMA IDENTIFICADAS (via grid search com AIC)")
print("="*70)

for serie_name in series_names:
    order, seasonal_order = sarima_orders[serie_name]
    results = fitted_models[serie_name]
    print(f"\n{serie_name}:")
    print(f"  Ordem SARIMA: {order}{seasonal_order}")
    print(f"  AIC: {results.aic:.2f}")
    print(f"  Descrição: SARIMA(p={order[0]}, d={order[1]}, q={order[2]})(P={seasonal_order[0]}, D={seasonal_order[1]}, Q={seasonal_order[2]}, m=7)")

print("\n" + "="*70)
print("Previsões salvas no formato CSV (será integrado com baselines)")
print("="*70)